In [1]:
import casadi as cas
import numpy as np
import json
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrow
from scipy.optimize import linear_sum_assignment

from typing import List, Dict

In [21]:
def load_formation(name):
    with open('3robots_patterns.json', 'r') as f:
        formations = json.load(f)
    return formations[name]

In [22]:
def assign_targets(robots, goal_positions):
    # create distance matrix
    n = len(robots)
    distances = np.zeros((n, n))

    for i, robot in enumerate(robots):
        for j, goal_position in enumerate(goal_positions):
            dist = np.sqrt( (robot.x - goal_position[0])**2 +
                        (robot.y - goal_position[1])**2 )
            distances[i][j] = dist**2
        
    # do target assignement
    assigned_positions = linear_sum_assignment(distances, maximize=False)
    for robot_idx, goal_idx in zip(assigned_positions[0], assigned_positions[1]):
        robot = robots[robot_idx]
        goal = goal_positions[goal_idx]

        robot.set_goal(goal[0], goal[1], goal[2])

    return robots

In [23]:
class Turtlebot:
    def __init__(self, x: float, y: float, w: float):
        self.x = x
        self.y = y
        self.w = w

    def move(self, v: float, w: float, dt: float):
        self.x = self.x + v * np.cos(w) * dt
        self.y = self.y + v * np.sin(w) * dt
        self.w = self.w + w * dt

    def set_goal(self, x, y, w):
        self.goal = [x, y, w]

    def plot(self, ax, color='blue', arrow_length=0.1, width=0.005, head_width=0.02):
        """
        dx = arrow_length * np.cos(self.w)
        dy = arrow_length * np.sin(self.w)

        arrow = FancyArrow(
            self.x,
            self.y,
            dx,
            dy,
            length_includes_head=True,
            width=width,
            head_width=head_width
        )
        ax.add_patch(arrow)
        """
        ax.scatter(self.x, self.y, color=color)

In [24]:
def dynamic_system(x, u, x_other, dt):
    # compute next position for current robot
    x_pos_next = x[0] + dt * u[0] * cas.cos(x[2])
    y_pos_next = x[1] + dt * u[0] * cas.sin(x[2])
    w_next = x[2] + dt * u[1]

    x_next = cas.vertcat(x_pos_next, y_pos_next, w_next)

    # compute next positions for all other robots using
    # constant velocity approach.
    s = 1 # set speed to one TODO: finetune
    x_others_next = []
    for state in x_other:
        x_pos_next = state[0] + dt * s * cas.cos(state[2])
        y_pos_next = state[1] + dt * s * cas.sin(state[2])
        w_next = state[2]
        next_state = cas.vertcat(x_pos_next, y_pos_next, w_next)
        x_others_next.append(next_state)

    return x_next, x_others_next

In [67]:
def cost_function(x, u, x_ref, u_ref, x_other):
    """
    Define the system of the optimal controll problem

    :param x: State vector for the current turtlebot [x, y, w] for T+1 timestamps.
    :param u: Controll input vector [v, w] for T timestamps
    """
    Qx = 0.5
    Ru = 0.5
    Qc = 0.5
    K = 3
    r = 1

    def Jx(x, x_ref):
        return Qx * cas.norm_2(x - x_ref)**2

    def Ju(u, u_ref):
        return Ru * cas.norm_2(u - u_ref)**2
        
    def Jc(x, x_other):
        sum = 0
        for x_agent in x_other:
            sum += (
                Qc / (
                    1 + cas.exp(
                        K * (cas.norm_2(x[:2] - x_agent[:2]) - r)
                    )
                )
            )
        return sum

    def JT(pos, target_pos):
        #return dist(pos, target_pos)
        return 0

    return Jx(x, x_ref) + Ju(u, u_ref) + Jc(x, x_other)

In [68]:
# load starting and target positions
x_init = load_formation('circle')
x_init = [x + [0] for x in x_init]

target_positions = load_formation('triangle')
target_positions = [x + [0] for x in target_positions]

# Initialize robots
robots = []
for x in x_init:
    r = Turtlebot(x[0], x[1], x[2])
    robots.append(r)

robots = assign_targets(robots, target_positions)

In [73]:
# Set up the optimization problem
max_linear_vel = 0.12 # m/s
max_angular_vel = 2.84 # rad/s

T = 3  # Number of time steps
dt = 0.1  # Time step (s)
N = 10 # number of other agents
iterations = 5

current_states = x_init

# Iterate over timesteps
for a in range(iterations):
    print(f" ============ Iteration {a} ================")

    # Solve optimization problem for each robot
    for j, robot in enumerate(robots):
        print(f"------- Robot {j} ----------")
        opti = cas.Opti()

        # initial and target state
        x_init = [robot.x, robot.y, robot.w]
        x_final = robot.goal
    
        # Define optimization variables
        x = opti.variable(3, N+1)  # robot state
        u = opti.variable(2, N)    # control inputs
        x_others = [opti.variable(3, N+1) for i in range(len(robots)-1)] # other robot states

        # make casadi reference target state
        x_ref = opti.variable(len(x_final))

        # make casadi reference controll state
        u_ref = opti.variable(2)

        # define future states
        for t in range(T):
            other_states = [states[:,t] for states in x_others]
            x_next, x_others_next = dynamic_system(x[:,t], u[:,t], other_states, dt)

            # constraint for next state
            opti.subject_to(x[:,t+1] == x_next)

            # constraints for next states for other agents
            for x_other, x_other_next in zip(x_others, x_others_next):
                opti.subject_to(x_other[:, t+1] == x_other_next)

        # add constraints
        # agent initial positiongues
        opti.subject_to(x[:,0] == x_init)
        opti.subject_to(u_ref == [0, 0])

        # initial positions for all other agents
        l= 0
        for k, other_robot in enumerate(robots):
            if j == k:
                # not for the current robot
                pass
            else:
                opti.subject_to(x_others[l][:,0] == [other_robot.x, other_robot.y, other_robot.w])
                l += 1

        # constraints for controll signals

        # for linear velocity
        opti.subject_to(
            opti.bounded(-max_linear_vel, u[0], max_linear_vel)
        )

        # for angular velocity
        opti.subject_to(
            opti.bounded(-max_angular_vel, u[1], max_angular_vel)
        )

        # define cost
        cost = 0
        for t in range(T):
            cost += cost_function(x[:,t], u[:,t], x_ref, u_ref, x_others)

        # tell casadi to minimize cost
        opti.minimize(cost)

        # solve system\
        solver_opts = {'ipopt.print_level': 0, 'print_time': 0}
        opti.solver('ipopt', solver_opts)
        solution = opti.solve()

        print(solution)

 ============ Iteration 0 ================
------- Robot 0 ----------


CasADi - 2024-04-11 16:47:21 WARNING("solver:nlp_grad_f failed: NaN detected for output grad_f_x, at (row 0, col 0).") [.../casadi/core/oracle_function.cpp:377]
CasADi - 2024-04-11 16:47:21 WARNING("solver:nlp_grad_f failed: NaN detected for output grad_f_x, at (row 0, col 0).") [.../casadi/core/oracle_function.cpp:377]


RuntimeError: Error in Opti::solve [OptiNode] at .../casadi/core/optistack.cpp:157:
.../casadi/core/optistack_internal.cpp:996: Assertion "return_success(accept_limit)" failed:
Solver failed. You may use opti.debug.value to investigate the latest values of variables. return_status is 'Invalid_Number_Detected'

In [74]:
opti.debug.value(x)

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [63]:
x

MX(opti202_x_1)